#### Canadian Traditional Media Audience Measurement - English vs. French Preferences

##### **Inspection and Cleaning**

This notebook details the methods used to inspect and clean the ["Audience Measurement - Current Trends"](https://open.canada.ca/data/dataset/3793028e-96ff-43cc-9281-bfcd83a114a8?utm_source=copilot.com) dataset from the official [open data archives](https://www.canada.ca/en/services/science/open-data.html) of the Government of Canada. The cleaned datasets will then be used to create visualizations to better understand patterns in French and English viewing and listening habits across Canada over time.

This analysis utilizes three datasets:
- ```Audiovisual.csv```, which consists of tuning data collected from newer audiovisual streaming platforms and services

- ```Radio.csv```, which consists of radio tuning data collected from traditional radio stations

- ```Television.csv```, which consists of TV tuning data collected from traditional TV stations 

Data cleaning techniques using pandas will be applied to each dataset to prepare the data for further transformation and analysis.


**Step 1:** Import the necessary libraries and load each .csv file into a dataframe.

In [15]:
import pandas as pd

audiovisual = pd.read_csv(r"C:\Users\rlynj\OneDrive\Documents\Project-2\data\original\Audiovisual.csv")
radio = pd.read_csv(r"C:\Users\rlynj\OneDrive\Documents\Project-2\data\original\Radio.csv")
television = pd.read_csv(r"C:\Users\rlynj\OneDrive\Documents\Project-2\data\original\Television.csv")

**Step 2:** Use pandas to inspect each data frame, noting inconsistencies and quality issues.

> Note: Null values in all three datasets are recorded as "-" with various spacings. Once the data is converted to the correct data type for each column, "-" will be converted to "NaN" and the prevalance and distribution of null values will become more clear. 


**Dataset 1 - Audiovisual**

In [ ]:
#Find out how the data is structured
audiovisual.head(20)

#Size of the dataframe
audiovisual.shape

#Get a look at column names
audiovisual.columns

#Understand data types for each column
audiovisual.info()

#Understand the categories in different categorical variables
audiovisual["Service Type"]
audiovisual["Market"]

*Notes from analysis:*

- Data is organized by year and quarter, with four quarters for each year 2023-2025
- 110 rows, 5 columns
- Column names include:
    - ```"Broadcast Year"```: The year in which the broadcast was measured
    - ```"Broadcast Quarter"```: The quarter (3-month period) in which the broadcast was measured
    - ```"Service Type"```: The type of service measured (e.g. "Linear TV")
    - ```"Market"```: Location of the measurement
    - ```"Total Quarterly Hours"```: Cumulative hours of watching per quarter
- Column names can be converted to snake_case for clarity and to avoid analysis complications
- ```"Total Quarterly Hours"``` column name has whitespace to remove
- ```"Broadcast Quarter"``` can be converted from 'Quarter X' to just an integer for analysis purposes
- ```"Total Quarterly Hours"``` needs to be converted from "string" to "integer" type
- ```"Service Type"``` contains 5 subcategories:
    - **"Broadcaster Video on Demand":** content made available "on-demand" by TV broadcasters
    - **"Free Ad-Supported TV":** Free to watch, ad-supported linear and on-demand platforms
    - **"Linear TV":** real time television that broadcasts scheduled programs
    - **"Social Media":** mobile and online video streaming services
    - **"Subscription Video on Demand & Other":** unlimited streaming service for a flat monthly rate
- This dataset only included two markets: "Ontario" and "Quebec"

**Dataset 2 - Radio**



In [ ]:
#Find out how the data is structured
radio.head(20)

#Size of the dataframe
radio.shape

#See column names
radio.columns

#See datatypes for each column
radio.info()

#See categorical values for different categorical variables
radio["Market"].unique()
radio["Service Type"].unique()
radio["Broadcast Language"].unique()

*Notes from analysis:*

- Data is also structured by year and quarter, with four quarters for each year 
- ```Radio.csv``` notably contains more data than ```Audiovisual.csv```, including years 2019-2025
- 810 rows, 7 columns
- Contains all the same columns as ```Audiovisual.csv``` with the addition of  ```"Broadcast Language"``` and ```"Universe"```
  - ```"Universe"``` describes the total estimated population of each region. This estimate is then used to scale findings from the sample to the broader population.
  - Although important, the same population value is recorded for each row per market. A seperate table  could be produced to summarize ```"Universe"``` population estimates for each market in each year and quarter.
- ```"Market"``` column contains extra whitespace
- Convert all columns to snake_case as performed above
- ```"Broadcast Quarter"``` can be converted to "integer" type, as performed above
- ```"Total Quarterly Hours"``` should be converted from "string" to "integer" type
- Create a seperate dataframe for ```"Universe"``` values to address duplicates
- Contains 5 market subcategories: "Calgary", "Edmonton", "Montreal", "Toronto" and "Vancouver"
- Contains 3 ```"Service Type"``` subcategories: 
  - **"CBC/SRC":** Government funded public broadcasting services
  - **"Non-commercial":** educational programming that does not generate revenue through advertisement
  - **"Private commercial":** programming privately owned by corporate media, generating revenue through advertisements




**Dataset 3 - Television**

In [ ]:
#Find out how the data is structured
television.head(20)

#Size of the dataframe
television.shape

#See column names
television.columns

#See datatypes for each column
television.info()

#See categorical values for different categorical variables
television["Market"].unique()
television["Service Type"].unique()
television["Broadcast Language"].unique()

*Notes from analysis*:

- Data is structured very similarly to ```Radio.csv``` , with the same 7 columns (```"Broadcast Year"```, ```"Broadcast Quarter"```, ```"Market"```, ```"Service Type"```, ```"Broadcast Language"```, ```"Total Quarterly Hours"``` and ```"Universe"```)
- 777 rows 
- ```"Quarter"``` can be converted from "string" to "integer" type, as performed above
- ```"Total Quarterly Hours"``` should be converted from "string" to "integer" type, as performed above 
- ```"Universe"``` values should exist in a separate table, as performed above
- Contains 5 ```"Market"``` subcategories: "Atlantic", "Central", "Ontario", "Quebec" and "West"
- Contains 3 ```"Service Type"``` categories:
    - **"Conventional":** national broadcast television services (containing a significant amount of local programming)
    - **"Discretionary & On Demand":** services open to subscribers (pay-per-view, sports subscriptions, on-demand services)
    - **"Non-Canadian & Other":** Both conventional and discretionary services originating outside of Canada
- ```"Broadcast Language"``` column contains "Other" observations, which will be discared for this analysis

**Step 3:** Apply pandas data manipulation techniques to each dataframe to correct the noted errors. Take care to note null value patterns and identify any outliers.

**Dataset 1 - Audiovisual**

In [16]:
#Remove whitespace and convert column names to snake_case 
audiovisual.columns = [column_name.strip().replace(" ","_") for column_name in audiovisual.columns]

#Convert all "-" with various spacing to pd.NA
audiovisual = audiovisual.replace(r"^\s*-\s*$", pd.NA, regex=True)

#Convert Broadcast_Quarter to int
audiovisual["Broadcast_Quarter"] = (audiovisual["Broadcast_Quarter"]
                                    .str.strip()
                                    .str.extract(r"(\d)$")
                                    .astype(int)
)

#Convert Total_Quarterly_Hours to int
audiovisual["Total_Quarterly_Hours"] = (audiovisual["Total_Quarterly_Hours"]
                                        .str.replace(",","")
                                        .str.strip()
                                        .astype(int)
)

#Check for outliers
audiovisual_sorted = audiovisual["Total_Quarterly_Hours"].dropna().sort_values(ascending=False)


**Dataset 2 - Radio**

In [17]:

#Remove whitespace from all columns and convert to snake_case
radio.columns = [column_name.strip().replace(" ","_") for column_name in radio.columns]

#Convert all "-" with various spacing to pd.NA
radio = radio.replace(r"^\s*-\s*$", pd.NA, regex=True)

#Convert "Broadcast_Quarter" to "int" type
radio["Broadcast_Quarter"] = (radio["Broadcast_Quarter"]
                              .str.strip()
                              .str.extract(r"(\d)$")
                              .astype(int)
)
#Convert "Total_Quarterly_Hours" to "int" type
radio["Total_Quarterly_Hours"] = (radio["Total_Quarterly_Hours"]
                                  .astype(str)
                                  .str.strip()
                                  .str.replace(",","")
                                  .astype("Int64")
)
                                        

#Create a smaller sub-dataframe for "Universe" values
#(First convert "Universe" to "int" type)
radio["Universe"] = (radio["Universe"]
                     .str.strip()
                     .str.replace(",","")
                     .astype("Int64")
)

#(Then create sub-dataframe)
radio_universe = (radio.groupby(["Broadcast_Year","Broadcast_Quarter","Market",])
            .agg({"Universe": "first"})
            .reset_index()
)

#(Then drop "Universe" from main dataframe)
radio = radio.drop(columns="Universe")

#Check for outliers
radio_sorted = radio["Total_Quarterly_Hours"].dropna().sort_values(ascending=False)

**Dataset 3 - Television**

In [18]:
#Remove whitespace from all columns and convert to snake_case
television.columns = [column_name.strip().replace(" ","_") for column_name in television.columns]

#Convert all "-" with various spacing to pd.NA
television = television.replace(r"^\s*-\s*$", pd.NA, regex=True)

#Convert "Broadcast_Quarter" to "int" type
television["Broadcast_Quarter"] = (television["Broadcast_Quarter"]
                                   .str.strip()
                                   .str.extract(r"(\d)$")
                                   .astype(int)
)

#Convert "Total_Quarterly_Hours" to "int" type
television["Total_Quarterly_Hours"] = (television["Total_Quarterly_Hours"]
                                       .str.strip()
                                       .str.replace(",","")
                                       .astype("Int64")
)

#Create a smaller sub-dataframe for "Universe" values
#(First convert "Universe" to "int" type)
television["Universe"] = (television["Universe"]
                     .str.strip()
                     .str.replace(",","")
                     .astype("Int64")
)

#(Then create sub-dataframe)
television_universe = (television.groupby(["Broadcast_Year", "Broadcast_Quarter", "Market"])
               .agg({"Universe" : "first"})
               .reset_index()
)

#(Then drop "Universe" from main dataframe)
television = television.drop(columns="Universe")

#Check for outliers
television_sorted = television["Total_Quarterly_Hours"].sort_values(ascending=False)

#Discard "Other" observations in "Broadcast_Language"
television = television[television["Broadcast_Language"] != "Other"]

**Step 4:** Validate each dataframe to make sure changes were applied correctly.

In [ ]:
audiovisual.head()
audiovisual.columns
audiovisual.dtypes
audiovisual.isna().sum()

radio.head()
radio.columns
radio.dtypes
radio.isna().sum()
print(radio_sorted)

radio_universe.head()
radio_universe.dtypes

television.head()
television.columns
television.dtypes
any(television["Broadcast_Language"] == "Other")
print(television_sorted)

television_universe.head()
television_universe.dtypes

>Note: The conversion of ```"Total Quarterly Hours"``` from string to integer data type was only partially successful for ```radio_cleaned.csv```. Integers always become float values (for example, "722" becomes "722.0") during the reading of the file when null values are present. Although the column could not cleanly be converted to just integers, float64 values are still numeric and suitable to use for visualization purposes.

**Step 5:** Save as cleaned files.


In [19]:
audiovisual.to_csv(r"C:\Users\rlynj\OneDrive\Documents\Project-2\data\cleaned\audiovisual_cleaned.csv", index=False)
radio.to_csv(r"C:\Users\rlynj\OneDrive\Documents\Project-2\data\cleaned\radio_cleaned.csv", index=False)
television.to_csv(r"C:\Users\rlynj\OneDrive\Documents\Project-2\data\cleaned\television_cleaned.csv", index=False)

radio_universe.to_csv(r"C:\Users\rlynj\OneDrive\Documents\Project-2\data\cleaned\universe_tables\radio_universe.csv")
television_universe.to_csv(r"C:\Users\rlynj\OneDrive\Documents\Project-2\data\cleaned\universe_tables\television_universe.csv")

These datasets were further utilized in the [visualizations.ipynb]() notebook to generate an interactive map visual and a trend line chart.